# Notebook 17 — Extended Qualitative Text-Case Analysis
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Revision note:** this version has been checked against your actual `src/config.py`. All file
paths, column names, and constants now import directly from your config. One correction from the
previous version: your config defines `DESC_FALLBACK_MODEL = 'D'` — the 37 games with no
description text never get routed to Model E in your real router, regardless of Model D's
confidence. Section 4 below now excludes these games from the "consulted" set, matching your
actual production behaviour exactly, since we have the real description text available here.

**Purpose:** Extend NB06's 4-game LIME qualitative sample into a systematic, categorised
analysis across a much larger set of the test-set games where the uncertainty router actually
consults Model E. This answers Reviewer 2's comment: *"Text fusion negative result lacks deeper
investigation (e.g., qualitative analysis of cases where text helps vs. hurts)."*

**What this adds beyond NB06:** rather than four hand-picked examples, this notebook
quantifies, across every consulted game, how often bringing in Model E actually **changes the
outcome** (flips a wrong Model D prediction to correct, or flips a correct one to wrong), then
runs word-level LIME explanations on a representative sample from each category, not just
disagreement cases.

**This notebook DOES need the raw dataset**, specifically `short_description` (`COL_DESC` in
your config), since the raw description text is not part of the numeric T4/T5 feature matrix.
The same row-alignment verification pattern from NB15/NB16 is reused before trusting that the
extracted text lines up with the correct test-set games.

**This notebook is READ-ONLY with respect to existing files.**

**Inputs (read-only), all sourced from `src.config`:**
- `RAW_CSV`, `COL_LABEL`, `COL_FILTER`, `COL_DESC`
- `FEATURES_T4_CSV`, `TARGET_COL`
- `MODEL_PATHS['D']`, `MODEL_PATHS['E']`, `PCA_PATH`
- `UNCERTAINTY_THRESHOLD`

---
## Contents
1. Setup and Imports
2. Load Raw Dataset and Verify Row Alignment (reused pattern from NB15/NB16)
3. Reconstruct the Test Split and Attach Description Text
4. Identify Consulted Games (excluding the no-description fallback) and Categorise Outcomes
5. Category Breakdown (quantitative — the main new evidence)
6. Word-Level LIME Explanations on a Representative Sample
7. Summary


## 1. Setup and Imports

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_CSV, COL_LABEL, COL_FILTER, COL_DESC, FEATURES_T4_CSV, TARGET_COL,
    RANDOM_STATE, MODEL_PATHS, PCA_PATH, UNCERTAINTY_THRESHOLD, DESC_FALLBACK_MODEL,
)

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    print('sentence-transformers not installed: pip install sentence-transformers --break-system-packages')
    SBERT_AVAILABLE = False

try:
    from lime.lime_text import LimeTextExplainer
    LIME_AVAILABLE = True
except ImportError:
    print('lime not installed: pip install lime --break-system-packages')
    LIME_AVAILABLE = False

N_LIME_SAMPLES_PER_CATEGORY = 5   # how many cases per category to run full LIME on

print(f'Project root          : {PROJECT_ROOT}')
print(f'Random state          : {RANDOM_STATE}')
print(f'Target column         : {TARGET_COL}')
print(f'Description column    : {COL_DESC}')
print(f'Uncertainty threshold  : {UNCERTAINTY_THRESHOLD}')
print(f'Desc fallback model    : {DESC_FALLBACK_MODEL}')
print(f'sentence-transformers available: {SBERT_AVAILABLE}')
print(f'lime available: {LIME_AVAILABLE}')


Project root          : C:\Users\3214h\Documents\fyp-steam-reception
Random state          : 42
Target column         : reception_label
Description column    : short_description
Uncertainty threshold  : 0.63
Desc fallback model    : D
sentence-transformers available: True
lime available: True


## 2. Load Raw Dataset and Verify Row Alignment

Same safety check as NB15/NB16: reconstruct the 75% label from `pct_pos_total` and confirm it
matches `games_features_t4.csv` row-for-row before trusting that `short_description` (extracted
in the same row order) lines up correctly with the T4 feature rows.


In [2]:
assert RAW_CSV.exists(), f'Not found: {RAW_CSV}'

df_raw = pd.read_csv(RAW_CSV, low_memory=False)
assert COL_DESC in df_raw.columns, f'{COL_DESC} column not found — check raw CSV schema.'

df_filtered = df_raw[
    (df_raw[COL_LABEL] != -1) &
    (df_raw[COL_FILTER] != -1) &
    (df_raw[COL_FILTER] >= 100)
].reset_index(drop=True)

print(f'After filtering: {df_filtered.shape[0]:,} rows (expected 20,383)')

df_filtered['reception_label_75_reconstructed'] = (df_filtered[COL_LABEL] >= 75).astype(int)

assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV}'
df_t4 = pd.read_csv(FEATURES_T4_CSV)

VERIFICATION_PASSED = False
if df_t4.shape[0] == df_filtered.shape[0]:
    match_rate = (df_filtered['reception_label_75_reconstructed'].values == df_t4[TARGET_COL].values).mean() * 100
    print(f'Row-for-row label agreement: {match_rate:.2f}%')
    VERIFICATION_PASSED = match_rate > 99.5

if VERIFICATION_PASSED:
    print('VERIFIED — safe to proceed with short_description alignment.')
else:
    print('*** WARNING: verification failed — DO NOT proceed. See NB15 Section 3 for troubleshooting.')


After filtering: 20,383 rows (expected 20,383)
Row-for-row label agreement: 100.00%
VERIFIED — safe to proceed with short_description alignment.


## 3. Reconstruct the Test Split and Attach Description Text

Same index-based split reconstruction as NB12 Part B — this guarantees the exact same test set
NB04 evaluated on, with description text attached in matching order.


In [3]:
assert VERIFICATION_PASSED, 'Verification failed in Section 2 — resolve before continuing.'

# Uses config.py's TIER_FEATURES[4] directly — verified via verify_tier_order.py.
from src.config import TIER_FEATURES
feature_cols = TIER_FEATURES[4]
missing = [c for c in feature_cols if c not in df_t4.columns]
assert not missing, f'Columns missing from CSV: {missing}'
X = df_t4[feature_cols].values
y = df_t4[TARGET_COL].values

all_idx = np.arange(len(df_t4))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_test = X[test_idx]
y_test = y[test_idx]
descriptions_test = df_filtered.loc[test_idx, COL_DESC].fillna('').values

n_empty = (descriptions_test == '').sum()
print(f'Test set: {len(X_test):,} games')
print(f'Descriptions attached: {len(descriptions_test):,}')
print(f'Empty/missing descriptions in test set: {n_empty} '
      f'(expected roughly {37 * 0.2:.1f} based on 37/20,383 project-wide)')


Test set: 4,077 games
Descriptions attached: 4,077
Empty/missing descriptions in test set: 6 (expected roughly 7.4 based on 37/20,383 project-wide)


## 4. Identify Consulted Games and Categorise Outcomes

For each test game, we compute Model D's confidence. A game is only actually routed to Model E
if **both** conditions hold: confidence falls below `UNCERTAINTY_THRESHOLD`, **and** the game has
a non-empty description. This matches your real router's `DESC_FALLBACK_MODEL = 'D'` behaviour —
games with no description text always fall back to Model D alone, regardless of confidence.
Games excluded by the fallback are reported separately, not silently dropped.


In [4]:
assert MODEL_PATHS['D'].exists() and MODEL_PATHS['E'].exists() and PCA_PATH.exists()

model_d = joblib.load(MODEL_PATHS['D'])
model_e = joblib.load(MODEL_PATHS['E'])
pca     = joblib.load(PCA_PATH)

assert SBERT_AVAILABLE, 'sentence-transformers required for this section.'
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding test-set descriptions with SBERT (one-off, may take a minute on CPU)...')
raw_embeddings_test = sbert_model.encode(list(descriptions_test), show_progress_bar=True)
embeddings_pca_test = pca.transform(raw_embeddings_test)
X_t5_test = np.hstack([X_test, embeddings_pca_test])

prob_d_test = model_d.predict_proba(X_test)
confidence_test = prob_d_test.max(axis=1)

has_description = np.array([bool(d.strip()) for d in descriptions_test])
low_confidence = confidence_test < UNCERTAINTY_THRESHOLD

consulted_mask = low_confidence & has_description
fallback_excluded_mask = low_confidence & (~has_description)

print(f'\nLow-confidence games (below threshold): {low_confidence.sum():,}')
print(f'  Of these, actually consulted (has description) : {consulted_mask.sum():,}')
print(f'  Of these, fallback to Model D (no description)  : {fallback_excluded_mask.sum():,}')
print(f'\n(NB04 reports 953 consulted games, 23.4% of the test set — the count above may differ')
print(f'slightly from 953 since NB04\'s own figure may or may not have already accounted for the')
print(f'fallback. Either way, this notebook now matches your actual router logic exactly.)')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding test-set descriptions with SBERT (one-off, may take a minute on CPU)...


Batches:   0%|          | 0/128 [00:00<?, ?it/s]


Low-confidence games (below threshold): 953
  Of these, actually consulted (has description) : 952
  Of these, fallback to Model D (no description)  : 1

(NB04 reports 953 consulted games, 23.4% of the test set — the count above may differ
slightly from 953 since NB04's own figure may or may not have already accounted for the
fallback. Either way, this notebook now matches your actual router logic exactly.)


In [5]:
prob_e_test = model_e.predict_proba(X_t5_test)
pred_d_alone = np.argmax(prob_d_test, axis=1)

combined_prob = (prob_d_test + prob_e_test) / 2
pred_router_consulted = np.argmax(combined_prob, axis=1)

# Build the categorisation ONLY for games actually consulted (has description AND low confidence)
idx_consulted = np.where(consulted_mask)[0]

categories = []
for i in idx_consulted:
    d_correct = (pred_d_alone[i] == y_test[i])
    router_correct = (pred_router_consulted[i] == y_test[i])

    if d_correct and router_correct:
        cat = 'No change — both correct'
    elif (not d_correct) and (not router_correct):
        cat = 'No change — both incorrect'
    elif (not d_correct) and router_correct:
        cat = 'Text corrected an error'
    else:  # d_correct and not router_correct
        cat = 'Text introduced an error'
    categories.append(cat)

categories = np.array(categories)
print('Categorisation complete for all consulted games.')


Categorisation complete for all consulted games.


## 5. Category Breakdown (quantitative — the main new evidence)

In [6]:
from collections import Counter

counts = Counter(categories)
total = len(categories)

print('=== OUTCOME CATEGORIES AMONG MODEL-E-CONSULTED GAMES ===')
print(f'{"Category":<32} {"Count":>8} {"%% of consulted":>16}')
print('-' * 58)
for cat in ['Text corrected an error', 'Text introduced an error',
            'No change — both correct', 'No change — both incorrect']:
    c = counts.get(cat, 0)
    print(f'{cat:<32} {c:>8} {c/total*100:>15.1f}%%')

net_benefit = counts.get('Text corrected an error', 0) - counts.get('Text introduced an error', 0)
print()
print(f'Net corrections (corrected minus introduced): {net_benefit:+d} games')
print(f'This net figure is the mechanism behind the routers Macro F1 gain reported in NB04.')


=== OUTCOME CATEGORIES AMONG MODEL-E-CONSULTED GAMES ===
Category                            Count  %% of consulted
----------------------------------------------------------
Text corrected an error               153            16.1%%
Text introduced an error              108            11.3%%
No change — both correct              426            44.7%%
No change — both incorrect            265            27.8%%

Net corrections (corrected minus introduced): +45 games
This net figure is the mechanism behind the routers Macro F1 gain reported in NB04.


## 6. Word-Level LIME Explanations on a Representative Sample

Sample a handful of games from the two most informative categories ("text corrected an error"
and "text introduced an error") and run `LimeTextExplainer`, extending NB06's original 4-game
sample with a properly stratified selection rather than hand-picked examples.


In [7]:
assert LIME_AVAILABLE, 'lime required for this section.'

rng = np.random.default_rng(RANDOM_STATE)

def sample_from_category(cat_name, n):
    matching = idx_consulted[categories == cat_name]
    if len(matching) == 0:
        return np.array([], dtype=int)
    n = min(n, len(matching))
    return rng.choice(matching, size=n, replace=False)

sample_corrected  = sample_from_category('Text corrected an error', N_LIME_SAMPLES_PER_CATEGORY)
sample_introduced = sample_from_category('Text introduced an error', N_LIME_SAMPLES_PER_CATEGORY)

lime_sample_idx = np.concatenate([sample_corrected, sample_introduced])
print(f'LIME sample: {len(sample_corrected)} "corrected" cases + '
      f'{len(sample_introduced)} "introduced error" cases = {len(lime_sample_idx)} total')


LIME sample: 5 "corrected" cases + 5 "introduced error" cases = 10 total


In [8]:
class_names = ['Not Well Received', 'Well Received']
explainer = LimeTextExplainer(class_names=class_names)

def make_predict_fn(fixed_structured_features):
    """Returns a function LIME can call: text list -> class probabilities,
    holding this one game's structured features fixed and only varying text."""
    def predict_fn(text_list):
        embeddings = sbert_model.encode(text_list)
        embeddings_pca = pca.transform(embeddings)
        n = len(text_list)
        structured_repeated = np.tile(fixed_structured_features, (n, 1))
        X_t5_batch = np.hstack([structured_repeated, embeddings_pca])
        return model_e.predict_proba(X_t5_batch)
    return predict_fn

lime_results = []
for pos, i in enumerate(lime_sample_idx):
    cat = categories[np.where(idx_consulted == i)[0][0]]
    text = descriptions_test[i]
    if not text.strip():
        print(f'[{pos+1}/{len(lime_sample_idx)}] Skipping — empty description (should not happen, '
              f'since consulted_mask already excludes these — flag this if you see it).')
        continue

    predict_fn = make_predict_fn(X_test[i])
    try:
        exp = explainer.explain_instance(text, predict_fn, num_features=8, num_samples=200)
        top_words = exp.as_list()
        lime_results.append({'index': int(i), 'category': cat, 'text': text, 'top_words': top_words})
        print(f'[{pos+1}/{len(lime_sample_idx)}] Category: {cat}')
        print(f'  Description: "{text[:100]}..."')
        print(f'  Top word contributions: {top_words}')
        print()
    except Exception as e:
        print(f'[{pos+1}/{len(lime_sample_idx)}] LIME failed on this game: {e}')


[1/10] Category: Text corrected an error
  Description: "The Shattering is a first-person, story-driven, psychological thriller. Its defining features are th..."
  Top word contributions: [(np.str_('Shattering'), 0.08555518092802979), (np.str_('gameplay'), -0.040113443829693826), (np.str_('of'), -0.0324779810784331), (np.str_('features'), 0.029515523025422694), (np.str_('meant'), -0.024222490701216442), (np.str_('in'), -0.022738295304234706), (np.str_('driven'), -0.016920656389540663), (np.str_('each'), -0.0154282420763345)]

[2/10] Category: Text corrected an error
  Description: "Solve the puzzle for a chance to see a video. 'Anipuzzle Series'!..."
  Top word contributions: [(np.str_('Series'), -0.10380633561481044), (np.str_('video'), 0.09409331223250045), (np.str_('Anipuzzle'), 0.03291943924088027), (np.str_('puzzle'), 0.031211950614028792), (np.str_('the'), -0.024636976414718247), (np.str_('a'), 0.016072884150712875), (np.str_('for'), 0.010721754327774741), (np.str_('to'), 0.01026

## 7. Summary

The Section 5 category table is the primary new evidence: it quantifies, across every genuinely
consulted game rather than 4 hand-picked ones, how often text helps versus hurts, correctly
excluding the no-description fallback games that never reach Model E in your real router.
Suggested paper wording once you have the actual counts: *"Across the [N] test-set games where
the uncertainty router consulted Model E, text corrected an otherwise-incorrect Model D
prediction in [X] cases and introduced a new error in [Y] cases, a net gain of [X-Y] corrected
predictions, consistent with the router's overall Macro F1 improvement."*

The Section 6 LIME output gives concrete word-level examples from both directions (helped/hurt)
to illustrate the pattern qualitatively, extending Figure 4's original 4-game sample.

Nothing in NB01–NB16 or in `games_features_t4.csv` was modified by running this notebook.
